In [1]:
# =============================================================================
# INDIVIDUAL METHOD RUNNER
# Quick testing of a single method on a single dataset
# =============================================================================

# -----------------------------------------------------------------------------
# CONFIGURATION - CHANGE THESE VALUES
# -----------------------------------------------------------------------------

METHOD = "xgboost"           # Method to run (e.g., 'xgboost', 'catboost', 'tabpfn', 'mlp')
DATASET = "0005.base_modelisation"        # Dataset name (e.g., '0014.hmeq', '0001.gmsc')
TASK = "lgd"                  # Task type: 'pd' (classification) or 'lgd' (regression)

# -----------------------------------------------------------------------------
# FIXED SETTINGS (for quick testing)
# -----------------------------------------------------------------------------

ROW_LIMIT = 10000             # Limit rows for fast execution
MAX_EPOCHS = 15              # Max epochs for deep learning methods
CV_SPLITS = 5                # Number of folds for cross-validation
TUNE = True                 # No HPO or HPO? 
SEED = 42                    # Random seed
TEST_SIZE = 0.2              # Test set fraction
VAL_SIZE = 0.2               # Validation set fraction

# -----------------------------------------------------------------------------
# SETUP
# -----------------------------------------------------------------------------

import sys
from pathlib import Path
import pickle
import json
from datetime import datetime

# Add project root to path (notebook is in notebooks/ folder)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"\n{'='*60}")
print(f" Running: {METHOD} on {DATASET} ({TASK.upper()})")
print(f"{'='*60}")
print(f"  Row limit:  {ROW_LIMIT}")
print(f"  Max epochs: {MAX_EPOCHS}")
print(f"  CV splits:  {CV_SPLITS}")
print(f"  HPO:        {TUNE}")
print(f"{'='*60}\n")

# -----------------------------------------------------------------------------
# RUN METHOD
# -----------------------------------------------------------------------------

from src.methods.method_runner import run_talent_method, get_available_methods

# Show available methods
available = get_available_methods()
print(f"Available classical methods: {available['classical']}")
print(f"Available deep methods: {available['deep'][:10]}... ({len(available['deep'])} total)")
print()

# Run the method
results = run_talent_method(
    task=TASK,
    dataset=DATASET,
    test_size=TEST_SIZE,
    val_size=VAL_SIZE,
    cv_splits=CV_SPLITS,
    seed=SEED,
    row_limit=ROW_LIMIT,
    method=METHOD,
    max_epoch=MAX_EPOCHS,
    tune=TUNE,
    verbose=True,
)

# -----------------------------------------------------------------------------
# DISPLAY RESULTS
# -----------------------------------------------------------------------------

print(f"\n{'='*60}")
print(f" RESULTS")
print(f"{'='*60}")

for fold_id, fold_results in results.items():
    print(f"\nFold {fold_id}:")
    print(f"  Train time: {fold_results['train_time']:.2f}s")
    print(f"  Samples:    {len(fold_results['y_true'])}")
    
    if TASK == 'lgd':
        print(f"  Clipped:    {fold_results['n_clipped_below']} below, {fold_results['n_clipped_above']} above")
    
    print(f"\n  Metrics:")
    for metric_name, metric_value in fold_results['metrics'].items():
        if not (isinstance(metric_value, float) and metric_value != metric_value):  # Skip NaN
            print(f"    {metric_name:20s}: {metric_value:.4f}")

# -----------------------------------------------------------------------------
# SAVE RESULTS
# -----------------------------------------------------------------------------

# Create output directory
output_dir = PROJECT_ROOT / 'results' / 'individual_method_runner'
output_dir.mkdir(parents=True, exist_ok=True)

# Generate filename with timestamp
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f"{METHOD}_{DATASET}_{TASK}_{timestamp}"

# Save as pickle (full results)
pickle_path = output_dir / f"{filename}.pkl"
with open(pickle_path, 'wb') as f:
    pickle.dump(results, f)
print(f"\nResults saved to: {pickle_path}")

# Save summary as JSON (metrics only, for easy viewing)
summary = {
    'method': METHOD,
    'dataset': DATASET,
    'task': TASK,
    'timestamp': timestamp,
    'config': {
        'row_limit': ROW_LIMIT,
        'max_epochs': MAX_EPOCHS,
        'cv_splits': CV_SPLITS,
        'tune': TUNE,
        'seed': SEED,
    },
    'folds': {}
}

for fold_id, fold_results in results.items():
    summary['folds'][fold_id] = {
        'train_time': fold_results['train_time'],
        'n_samples': len(fold_results['y_true']),
        'metrics': {k: v for k, v in fold_results['metrics'].items() if not (isinstance(v, float) and v != v)},
    }
    if TASK == 'lgd':
        summary['folds'][fold_id]['n_clipped_below'] = fold_results['n_clipped_below']
        summary['folds'][fold_id]['n_clipped_above'] = fold_results['n_clipped_above']

json_path = output_dir / f"{filename}.json"
with open(json_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"Summary saved to: {json_path}")

print(f"\n{'='*60}")
print(f" DONE")
print(f"{'='*60}")

Project root: c:\Users\U0152019\OneDrive - KU Leuven\PhD Documents\Projects\1. TabPFN\TabPFNCredit

 Running: xgboost on 0005.base_modelisation (LGD)
  Row limit:  10000
  Max epochs: 15
  CV splits:  5
  HPO:        True

Available classical methods: ['LinearRegression', 'LogReg', 'NCM', 'NaiveBayes', 'RandomForest', 'catboost', 'dummy', 'knn', 'lightgbm', 'svm', 'xgboost']
Available deep methods: ['amformer', 'autoint', 'bishop', 'danets', 'dcn2', 'dnnr', 'excelformer', 'ftt', 'grande', 'grownet']... (38 total)


Running xgboost (classical) on 0005.base_modelisation (LGD)

[HPO] Per-fold hyperparameter optimization enabled
[HPO] Each fold: 50 trials (optimizes on validation loss)

Preparing data with 5 CV splits...
Fold IDs: [1, 2, 3, 4, 5]

Directory setup:
  Config directory: C:\Users\U0152019\OneDrive - KU Leuven\PhD Documents\Projects\1. TabPFN\TabPFNCredit\config_hpo\lgd\0005.base_modelisation\xgboost\HPO_PER_FOLD
  Merged config: xgboost-all-folds.json

Fold 1/5
using gpu: 0
{'

[I 2026-01-05 15:09:55,211] A new study created in memory with name: no-name-d0c0a4ed-6ed9-4121-877e-fe88d23da1cc


{'fit': {'verbose': False, 'n_bins': 2}, 'model': {'subsample': 0.8, 'colsample_bytree': 0.8, 'early_stopping_rounds': 50, 'booster': 'gbtree', 'n_estimators': 2000, 'n_jobs': -1, 'tree_method': 'hist'}}


  0%|          | 0/50 [00:00<?, ?it/s]

[0]	validation_0-rmse:0.93016
[1]	validation_0-rmse:0.90238
[2]	validation_0-rmse:0.88326
[3]	validation_0-rmse:0.87027
[4]	validation_0-rmse:0.85138
[5]	validation_0-rmse:0.83162
[6]	validation_0-rmse:0.81439
[7]	validation_0-rmse:0.79893
[8]	validation_0-rmse:0.79411
[9]	validation_0-rmse:0.79624
[10]	validation_0-rmse:0.78980
[11]	validation_0-rmse:0.78302
[12]	validation_0-rmse:0.77995
[13]	validation_0-rmse:0.77757
[14]	validation_0-rmse:0.78129
[15]	validation_0-rmse:0.78484
[16]	validation_0-rmse:0.78781
[17]	validation_0-rmse:0.78044
[18]	validation_0-rmse:0.78476
[19]	validation_0-rmse:0.78796
[20]	validation_0-rmse:0.78735
[21]	validation_0-rmse:0.78608
[22]	validation_0-rmse:0.78699
[23]	validation_0-rmse:0.78961
[24]	validation_0-rmse:0.78967
[25]	validation_0-rmse:0.79181
[26]	validation_0-rmse:0.79102
[27]	validation_0-rmse:0.79099
[28]	validation_0-rmse:0.79264
[29]	validation_0-rmse:0.79205
[30]	validation_0-rmse:0.79085
[31]	validation_0-rmse:0.78948
[32]	validation_0-

...

Best trial: 43. Best value: 0.26377:  98%|█████████▊| 49/50 [00:12<00:00,  5.69it/s]

[I 2026-01-05 15:11:18,124] Trial 48 finished with value: 0.2760829790582212 and parameters: {'optional_alpha': False, 'colsample_bylevel': 0.706397641704593, 'colsample_bytree': 0.7438678588144226, 'optional_gamma': True, 'gamma': 2.4193091294211136e-07, 'optional_lambda': True, 'lambda': 0.010641279356739822, 'learning_rate': 0.0447589907238361, 'max_depth': 3, 'min_child_weight': 0.0003355144363546751, 'subsample': 0.762408567539257, 'n_bins': 115}. Best is trial 43 with value: 0.2637699012765822.
[0]	validation_0-rmse:0.94677
[1]	validation_0-rmse:0.94676
[2]	validation_0-rmse:0.94676
[3]	validation_0-rmse:0.94676
[4]	validation_0-rmse:0.94675
[5]	validation_0-rmse:0.94674
[6]	validation_0-rmse:0.94674
[7]	validation_0-rmse:0.94673
[8]	validation_0-rmse:0.94673
[9]	validation_0-rmse:0.94673
[10]	validation_0-rmse:0.94672
[11]	validation_0-rmse:0.94672
[12]	validation_0-rmse:0.94671
[13]	validation_0-rmse:0.94670
[14]	validation_0-rmse:0.94670
[15]	validation_0-rmse:0.94670
[16]	val

Best trial: 43. Best value: 0.26377: 100%|██████████| 50/50 [00:12<00:00,  3.92it/s]

[I 2026-01-05 15:11:18,371] Trial 49 finished with value: 0.39420694321722644 and parameters: {'optional_alpha': True, 'alpha': 0.07953844690114673, 'colsample_bylevel': 0.7994924049308102, 'colsample_bytree': 0.7177155372262155, 'optional_gamma': True, 'gamma': 0.00106917032014155, 'optional_lambda': True, 'lambda': 0.0672811899103102, 'learning_rate': 1.1737868078048674e-05, 'max_depth': 5, 'min_child_weight': 0.012914611691062934, 'subsample': 0.8029158730154342, 'n_bins': 159}. Best is trial 43 with value: 0.2637699012765822.
Best Hyper-Parameters
{'model': {'alpha': 0.041882903871417534, 'colsample_bylevel': 0.7809190468770772, 'colsample_bytree': 0.6829542664009879, 'gamma': 1.0110420609348034e-08, 'lambda': 31.57079961412316, 'learning_rate': 0.09712787741002538, 'max_depth': 3, 'min_child_weight': 2.5295631960321654e-06, 'subsample': 0.9219702817320441}, 'fit': {'n_bins': 120}}
[HPO] Config saved (fold 5)
[HPO] Best hyperparameters: {'alpha': 0.041882903871417534, 'colsample_by


[CLIPPING] 5 predictions < 0, 0 > 1 (4.2% total)

Fold 5 metrics:
  R2: 0.4982
  MSE: 0.0852
  RMSE: 0.2918
  MAE: 0.2246
  MedAE: 0.1804
  MaxError: 0.9292
  Explained_Variance: 0.5002
  MAPE: 638.2678
  Pearson_Corr: 0.7083
  Spearman_Corr: 0.5732

Completed 5 folds for xgboost

[HPO] All fold configs saved: C:\Users\U0152019\OneDrive - KU Leuven\PhD Documents\Projects\1. TabPFN\TabPFNCredit\config_hpo\lgd\0005.base_modelisation\xgboost\HPO_PER_FOLD\xgboost-all-folds.json
[HPO] Per-fold hyperparameters optimized independently
[HPO] No data leakage - each fold optimized on its own data


 RESULTS

Fold 1:
  Train time: 0.65s
  Samples:    119
  Clipped:    0 below, 0 above

  Metrics:
    R2                  : 0.3875
    MSE                 : 0.1002
    RMSE                : 0.3166
    MAE                 : 0.2483
    MedAE               : 0.1809
    MaxError            : 0.8273
    Explained_Variance  : 0.3876
    MAPE                : 494.9541
    Pearson_Corr        : 0.6297
    S